# Tome 15, Appendix K: Randomness in the Integers (Python companion)

Exercises first, solutions at the end. We sieve the Liouville and Möbius functions up to $10^7$ and test them against the coin model (Proposition
11.1, Proposition 11.3), check the small cases of the Erdős discrepancy problem (Theorem 11.4, Proposition 11.5),
search for long multiplicative sequences of small discrepancy (Figure 11.2), test Artin's conjecture and Theorem 11.6, and compare class
numbers with the Cohen--Lenstra heuristics. We use `numpy` and `matplotlib`.

*Before each solution, predict what the output will look like.*

In [1]:
import math, time, itertools
import numpy as np
import matplotlib.pyplot as plt

## Exercises

**P11.1 (★★) Chowla.** Sieve $\lambda(n)$ for $n\le10^7$. Compute $\frac1x\sum_{n\le x}\lambda(n)\lambda(n+h)$ for $h=1,\dots,10$ at $x=10^7$, and the frequencies of
the eight sign patterns of $(\lambda(n),\lambda(n+1),\lambda(n+2))$.

**P11.2 (★) Möbius against periodic sequences.** Sieve $\mu$ and compute $\frac1x\sum_{n\le x}\mu(n)a(n)$ for $a(n)=\mathbf 1_{n\equiv r\,(12)}$, $r=0,\dots,11$, and
for $a(n)=\cos(2\pi n/7)$.

**P11.3 (★) Discrepancy by brute force.** Check Theorem 11.4 over all $2^{12}$ sequences, and Proposition 11.5 for $N\le10^5$.

**P11.4 (★★) Multiplicative sequences.** By backtracking over the values at primes, find the longest completely multiplicative $\pm1$ sequences with partial
sums in $[-1,1]$, then in $[-2,2]$, and count those of maximal length.

**P11.5 (★★) Artin.** For the primes $p\le10^7$, compute the proportion with primitive root $a$ for $a=2,3,5,10$, the proportion with primitive root $2$
among the $p\equiv\pm3\pmod8$ (Exercise 11.6), and the average of $\frac{\varphi(p-1)}{p-1}$ (Theorem 11.6).

**P11.6 (★★) Cohen--Lenstra.** Compute the class numbers $h(d)$ of the imaginary quadratic fields with $|d|\le2\cdot10^5$ by counting reduced forms, and
compare the proportions of $h$ divisible by $3$, $5$, $7$ with $1-\eta(p)$ (Exercise 11.8).

## Solutions

### P11.1
Start from $\lambda=1$ and flip the sign of every multiple of every prime power $p^k$: then $\lambda(n)=(-1)^{\Omega(n)}$.

In [2]:
N = 10**7
lam = np.ones(N + 11, dtype=np.int8)
sieve = np.ones(N + 11, bool); sieve[:2] = False
for p in range(2, int((N + 10) ** .5) + 1):
    if sieve[p]: sieve[p*p::p] = False
primes = np.nonzero(sieve)[0]
t0 = time.time()
for p in primes:
    q = int(p)
    while q <= N + 10:
        lam[q::q] *= -1
        q *= p
lam[0] = 0
print('sieve: %.1f s; lambda(1..12) =' % (time.time() - t0), list(lam[1:13]))
L = lam[1:N + 1].astype(np.int64)
print('L(10^7) =', L.sum())
for h in range(1, 11):
    print('h = %2d   (1/x) sum lambda(n)lambda(n+h) = %+.6f' % (h, np.dot(L, lam[1 + h:N + 1 + h].astype(np.int64)) / N))
codes = (lam[1:N + 1] > 0).astype(int) * 4 + (lam[2:N + 2] > 0).astype(int) * 2 + (lam[3:N + 3] > 0).astype(int)
freq = np.bincount(codes, minlength=8) / N
for c in range(8):
    print(''.join('+' if (c >> (2 - i)) & 1 else '-' for i in range(3)), '%.5f' % freq[c])

sieve: 1.2 s; lambda(1..12) = [np.int8(1), np.int8(-1), np.int8(-1), np.int8(1), np.int8(-1), np.int8(1), np.int8(-1), np.int8(-1), np.int8(1), np.int8(1), np.int8(-1), np.int8(-1)]


L(10^7) = -842


h =  1   (1/x) sum lambda(n)lambda(n+h) = -0.000205
h =  2   (1/x) sum lambda(n)lambda(n+h) = +0.000125
h =  3   (1/x) sum lambda(n)lambda(n+h) = -0.000318
h =  4   (1/x) sum lambda(n)lambda(n+h) = +0.000078
h =  5   (1/x) sum lambda(n)lambda(n+h) = -0.000209
h =  6   (1/x) sum lambda(n)lambda(n+h) = -0.000273
h =  7   (1/x) sum lambda(n)lambda(n+h) = +0.000511


h =  8   (1/x) sum lambda(n)lambda(n+h) = +0.000215
h =  9   (1/x) sum lambda(n)lambda(n+h) = -0.000126
h = 10   (1/x) sum lambda(n)lambda(n+h) = -0.000690


--- 0.12507
--+ 0.12492
-+- 0.12500
-++ 0.12505
+-- 0.12492
+-+ 0.12513
++- 0.12505
+++ 0.12486


All ten correlations are below $10^{-3}$ in absolute value, and the eight patterns all have frequency close to $\frac18=0.125$, as Proposition
11.1 predicts under Chowla's conjecture. The deviations are a few thousandths: larger than for a fair coin with $10^7$ tosses (about $10^{-4}$), a
reminder that $\frac1x\sum$ converges slowly for $\lambda$.

### P11.2
$\mu$ by the same kind of sieve: start from $1$, multiply by $-1$ at the multiples of $p$, and set $0$ at the multiples of $p^2$.

In [3]:
mu = np.ones(N + 1, dtype=np.int8)
for p in primes:
    p = int(p)
    if p > N: break
    mu[p::p] *= -1
    if p * p <= N: mu[p*p::p*p] = 0
mu[0] = 0
n = np.arange(N + 1)
print('M(10^7) =', int(mu.astype(np.int64).sum()))
print('(1/x) sum mu(n) 1[n = r mod 12], r = 0..11:')
print([round(float(mu[n % 12 == r].astype(np.int64).sum()) / N, 6) for r in range(12)])
print('(1/x) sum mu(n) cos(2 pi n/7) = %.2e' % (np.dot(mu.astype(float), np.cos(2*np.pi*n/7)) / N))

M(10^7) = 1037
(1/x) sum mu(n) 1[n = r mod 12], r = 0..11:


[0.0, -2.7e-05, 5.4e-05, 8e-06, 0.0, 3.3e-05, -1.5e-05, 1.3e-05, 0.0, 4e-05, 8e-06, -1e-05]


(1/x) sum mu(n) cos(2 pi n/7) = 3.13e-05


Every average is below $10^{-3}$ in absolute value; the classes $r$ with $\gcd(r,12)$ divisible by $4$ give exactly $0$, because then $4\mid n$ and $\mu(n)=0$.

### P11.3

In [4]:
def disc(x):
    m = 0
    for d in range(1, len(x) + 1):
        s = 0
        for k in range(d, len(x) + 1, d):
            s += x[k - 1]; m = max(m, abs(s))
    return m
for n in (11, 12):
    good = [x for x in itertools.product((1, -1), repeat=n) if disc(x) <= 1]
    print('length %d: %d sequences of discrepancy 1' % (n, len(good)), [''.join('+' if v > 0 else '-' for v in x) for x in good])

def f3(n):
    while n % 3 == 0: n //= 3
    return 1 if n % 3 == 1 else -1
def ones3(n):
    c = 0
    while n: c += (n % 3 == 1); n //= 3
    return c
S, ok, best = 0, True, 0
for m in range(1, 10**5 + 1):
    S += f3(m); ok &= (S == ones3(m)); best = max(best, S)
    if best >= 1 and m in (13, 40, 121, 364, 1093, 3280, 9841, 29524):
        print('N = %6d  discrepancy %d  floor(log_3(2N+1)) = %d' % (m, best, int(math.floor(math.log(2*m + 1, 3) + 1e-12))))
print('partial sums = number of ternary digits 1 for all N <= 10^5:', ok)

length 11: 4 sequences of discrepancy 1 ['+--+-++--++', '+--+-++--+-', '-++-+--++-+', '-++-+--++--']
length 12: 0 sequences of discrepancy 1 []
N =     13  discrepancy 3  floor(log_3(2N+1)) = 3
N =     40  discrepancy 4  floor(log_3(2N+1)) = 4
N =    121  discrepancy 5  floor(log_3(2N+1)) = 5
N =    364  discrepancy 6  floor(log_3(2N+1)) = 6
N =   1093  discrepancy 7  floor(log_3(2N+1)) = 7
N =   3280  discrepancy 8  floor(log_3(2N+1)) = 8
N =   9841  discrepancy 9  floor(log_3(2N+1)) = 9
N =  29524  discrepancy 10  floor(log_3(2N+1)) = 10
partial sums = number of ternary digits 1 for all N <= 10^5: True


Four sequences of length $11$, none of length $12$; the partial sums of the character-like function count ternary digits $1$, and the discrepancy reaches
$k$ exactly at $N=\frac{3^k-1}2$.

### P11.4
For a completely multiplicative $f$ the discrepancy is the largest $|f(1)+\dots+f(k)|$, and $f(n)$ is forced at composite $n$: we branch only at primes.

In [5]:
def search(C, N=400):
    spf = list(range(N + 1))
    for i in range(2, N + 1):
        if spf[i] == i:
            for j in range(i * i, N + 1, i):
                if spf[j] == j: spf[j] = i
    f = [0] * (N + 1); f[1] = 1
    best, found = [0], {}
    def go(n, s):
        if n - 1 >= best[0]:
            if n - 1 > best[0]: best[0] = n - 1; found.clear()
            found[tuple(f[1:n])] = 1
        if n > N: return
        ch = (1, -1) if spf[n] == n else (f[spf[n]] * f[n // spf[n]],)
        for v in ch:
            if abs(s + v) <= C:
                f[n] = v; go(n + 1, s + v)
        f[n] = 0
    go(2, 1)
    return best[0], list(found)
for C in (1, 2):
    t0 = time.time(); L_, seqs = search(C)
    print('C = %d: longest length %d, %d sequences of that length (%.2f s)' % (C, L_, len(seqs), time.time() - t0))
    print('  first:', ''.join('+' if v > 0 else '-' for v in seqs[0][:60]), '...' if L_ > 60 else '')

C = 1: longest length 9, 1 sequences of that length (0.00 s)
  first: +--+-++-+ 
C = 2: longest length 246, 500 sequences of that length (0.01 s)
  first: +--+-+--+++--+++--+-+--+++--+-+--+++--++---+-+--+-+-++-+--++ ...


Length $9$ for discrepancy $1$ (one sequence, Exercise 11.5) and $246$ for discrepancy $2$ ($500$ sequences, Figure 11.2). For sequences that
are not multiplicative the answer for discrepancy $2$ is $1160$ (Konev and Lisitsa): there is no forcing at composite $n$, and backtracking alone is hopeless.

### P11.5
$a$ is a primitive root modulo $p$ if and only if $a^{(p-1)/q}\not\equiv1$ for every prime $q\mid p-1$. A smallest-prime-factor table factors $p-1$ quickly.

In [6]:
M = 10**7
spf = np.zeros(M + 1, dtype=np.int32)
for p in primes:
    p = int(p)
    if p > M: break
    blk = spf[p::p]; blk[blk == 0] = p
def pfactors(m):
    out = []
    while m > 1:
        q = int(spf[m]); out.append(q)
        while m % q == 0: m //= q
    return out
P = [int(p) for p in primes if 2 < p <= M]
t0 = time.time()
facs = [pfactors(p - 1) for p in P]
for a in (2, 3, 5, 10):
    cnt = sum(1 for p, fs in zip(P, facs) if p % a and all(pow(a, (p - 1)//q, p) != 1 for q in fs))
    print('a = %2d: proportion %.5f' % (a, cnt / len(P)))
A = 1.0
for q in primes[:100000]: A *= 1 - 1/(int(q)*(int(q)-1))
r8 = [p for p, fs in zip(P, facs) if p % 8 in (3, 5)]
c8 = sum(1 for p, fs in zip(P, facs) if p % 8 in (3, 5) and all(pow(2, (p - 1)//q, p) != 1 for q in fs))
print('Artin constant A = %.7f, 20A/19 = %.5f;  among p = +-3 mod 8: %.5f (2A = %.5f)' % (A, 20*A/19, c8/len(r8), 2*A))
avg = np.mean([np.prod([1 - 1/q for q in fs]) for fs in facs])
print('average of phi(p-1)/(p-1) over odd p <= 10^7: %.5f   (%.0f s)' % (avg, time.time() - t0))

a =  2: proportion 0.37391


a =  3: proportion 0.37411


a =  5: proportion 0.39394


a = 10: proportion 0.37450


Artin constant A = 0.3739558, 20A/19 = 0.39364;  among p = +-3 mod 8: 0.74764 (2A = 0.74791)


average of phi(p-1)/(p-1) over odd p <= 10^7: 0.37400   (9 s)


The proportions for $a=2$, $3$ and $10$ are close to $A=0.37396$, the one for $a=5$ is close to $\frac{20}{19}A=0.39364$ (Tome 13 explains the correction),
the primes $\equiv\pm3\pmod8$ give about $2A$, and the average of $\frac{\varphi(p-1)}{p-1}$ is $0.37400$, very close to $A$ as Theorem 11.6 says.

### P11.6
A reduced form $(a,b,c)$ has $|b|\le a\le c$, with $b\ge0$ if $|b|=a$ or $a=c$, and discriminant $D=b^2-4ac$; for each $a$ and $b$ we add $1$ at all
$|D|=4ac-b^2\le X$ at once with `numpy`. For a fundamental discriminant every form is primitive, and $h(d)$ is the number of reduced forms.

In [7]:
X = 2 * 10**5
h = np.zeros(X + 1, dtype=np.int64)
for a in range(1, int((X / 3) ** .5) + 1):
    for b in range(-a + 1, a + 1):
        c0 = a
        cmax = (X + b * b) // (4 * a)
        if cmax < c0: continue
        Ds = 4 * a * np.arange(c0, cmax + 1) - b * b
        if b < 0: Ds = Ds[1:]                      # (a, b, a) with b < 0 is not reduced
        np.add.at(h, Ds, 1)
sqf = np.ones(X + 1, bool)
for p in range(2, int(X ** .5) + 1): sqf[p*p::p*p] = False
D = np.arange(X + 1)
fund = ((D % 4 == 3) & sqf) | ((D % 4 == 0) & sqf[D // 4] & np.isin((D // 4) % 4, [1, 2]))
fund[:3] = False
print('h(-3), h(-4), h(-23), h(-47), h(-71), h(-163) =', [int(h[k]) for k in (3, 4, 23, 47, 71, 163)])
eta = lambda p: np.prod([1 - p**-k for k in range(1, 60)])
for Y in (10**4, 10**5, 2 * 10**5):
    sel = fund & (D <= Y)
    hs = h[sel]
    print('|d| <= %6d: %6d fields;' % (Y, sel.sum()), '  '.join('P(%d | h) = %.4f' % (p, np.mean(hs % p == 0)) for p in (3, 5, 7)))
print('Cohen-Lenstra:', '  '.join('%.4f' % (1 - eta(p)) for p in (3, 5, 7)))

h(-3), h(-4), h(-23), h(-47), h(-71), h(-163) = [1, 1, 3, 5, 7, 1]
|d| <=  10000:   3043 fields; P(3 | h) = 0.3421  P(5 | h) = 0.2028  P(7 | h) = 0.1351
|d| <= 100000:  30392 fields; P(3 | h) = 0.3784  P(5 | h) = 0.2213  P(7 | h) = 0.1482
|d| <= 200000:  60800 fields; P(3 | h) = 0.3868  P(5 | h) = 0.2256  P(7 | h) = 0.1512
Cohen-Lenstra: 0.4399  0.2397  0.1632


The class numbers agree with the classical values ($h(-23)=3$, $h(-163)=1$). The proportions are below the predictions and climb slowly, as in the table of
the chapter (which goes to $2\cdot10^6$ with a compiled program); for $p=3$ the secondary term in the count of cubic fields explains the delay.